In [1]:
from langchain_community.document_loaders import PyPDFLoader
from dotenv import load_dotenv
load_dotenv()
import os

In [2]:
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
# Initialize a Pinecone client with your API key
api_key = os.environ.get("PINECONE_API_KEY")

In [3]:
pdf_path = "D:\\Data_analytics_new\\Sub_Resume_Building\\Legal_Lense\\data\\sample_rent.pdf"
pdf_path1 = "D:\\Data_analytics_new\\Sub_Resume_Building\\Legal_Lense\\data\\dushyant (1).pdf"

In [8]:
# Load PDF
loader = PyPDFLoader(pdf_path1)

documents = loader.load()

In [7]:
def is_text_empty(docs):
    total_text = " ".join([doc.page_content.strip() for doc in docs])
    return len(total_text) < 50  # threshold

In [14]:
import fitz  # PyMuPDF
import base64
from openai import OpenAI
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document

client = OpenAI()

def is_text_empty(docs):
    text = " ".join([d.page_content.strip() for d in docs])
    return len(text) < 50


def ocr_with_vision(pdf_path):
    import fitz
    import base64

    doc = fitz.open(pdf_path)
    ocr_texts = []

    for page in doc:
        pix = page.get_pixmap()
        img_bytes = pix.tobytes("png")
        img_base64 = base64.b64encode(img_bytes).decode()

        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "text",
                            "text": "Extract all text clearly. Preserve sections and clauses."
                        },
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": f"data:image/png;base64,{img_base64}"
                            }
                        }
                    ]
                }
            ]
        )

        ocr_texts.append(response.choices[0].message.content)

    return [Document(page_content=t) for t in ocr_texts]


def load_pdf_smart(pdf_path):
    loader = PyPDFLoader(pdf_path)
    docs = loader.load()

    if is_text_empty(docs):
        print("⚠️ Using Vision OCR fallback...")
        docs = ocr_with_vision(pdf_path)

    return docs

In [15]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=120
)

docs = load_pdf_smart(pdf_path1)
chunks = text_splitter.split_documents(docs)

⚠️ Using Vision OCR fallback...


In [16]:
chunks

[Document(metadata={}, page_content='Sure! Here’s the extracted text:\n\n---\n\n**RENT AGREEMENT**\n\nThis Rent Agreement is made at Gurugram on 05/01/2026 between:\n\n**Smti. Suresh Devi W/o Sh. Mahender Singh R/o RZ-687/A, Raj Nagar Part-1, Palam Colony, New Delhi-110045** (Hereinafter called the Lessor).\n\nWhere the terms and conditions (context so admit include its representatives, executors, administrator, successors and assigns).\n\nAND'),
 Document(metadata={}, page_content='AND\n\n**Mr. Dushyant Kumar Verma S/o Sh. Dashrath Lal Verma R/o H.No.36, School Para, Simga, Tildabandha, Baloda Bazar, Newari, Chhattisgarh, 493196** (Hereinafter referred to as the Lessee, where the terms and conditions so admits include its representatives, heirs, executors, administrators, successors and assigns).'),
 Document(metadata={}, page_content='Where the Lessor/lessor is the absolute owner of and agreed to give on rent the property at H.No.9-A, Gali No. G-1, G Block, Ashok Vihar Ph-III, Gurugr

#### Chunking

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""],  # Hierarchy of separators
    is_separator_regex=False,
)

chunks = recursive_splitter.split_documents(documents)

print(len(chunks))
print(chunks[0].page_content)

20
Teacher Resources for Consumer.gov | Developed for the FTC by the Center for Applied Linguistics 
SAMPLE RENTAL AGREEMENT
(Basic / Beginning)
THIS AGREEMENT made this 15th Day of June, 2012, by and between ABC Properties, herein called 
“Landlord,” and Silvia Mando, herein called “Tenant.” Landlord hereby agrees to rent to Tenant the dwelling 
located at 9876 Cherry Avenue, Apartment 426 under the following terms and conditions.
1. FIXED-TERM AGREEMENT (LEASE):


#### Store in Vector Database

In [18]:
# Import the Pinecone library
from pinecone import Pinecone

pc = Pinecone(api_key=api_key)

In [19]:
# Create a dense index with integrated embedding

index_name = "legal-lense-index1"  ## index name should be unique across your Pinecone project, and name doesn't have _ should be properly formatted

if not pc.has_index(name=index_name):
    pc.create_index(
    name=index_name,
    dimension=1536,  # IMPORTANT
    metric="cosine",
    spec={
        "serverless": {
            "cloud": "aws",
            "region": "us-east-1"
        }
    }
)

#### Hybrid Search + Reranking

In [20]:
### load the index 

from langchain_pinecone import PineconeVectorStore
from langchain_openai import OpenAIEmbeddings, ChatOpenAI

embedding_model = OpenAIEmbeddings()

vectorstore = PineconeVectorStore(
    index_name="legal-lense-index1",
    embedding=embedding_model
)



llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    n=3   # allow multiple generations
)

d:\Data_analytics_new\Sub_Resume_Building\Legal_Lense\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
import pandas as pd

df_eval = pd.read_excel("D:\\Data_analytics_new\\Sub_Resume_Building\\Legal_Lense\\data\\rag_eval_dataset.xlsx")

questions = df_eval["question"].tolist()
ground_truths = df_eval["ground_truth"].tolist()

In [21]:
from langchain_community.retrievers import BM25Retriever

# Create BM25 from your chunks
bm25_retriever = BM25Retriever.from_documents(chunks)

# Set how many docs BM25 returns
bm25_retriever.k = 10

In [22]:
vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

In [23]:
from collections import defaultdict

def hybrid_retrieve(query, top_k=5):
    # Step 1: BM25 raw scores
    bm25_results = bm25_retriever.invoke(query)
    bm25_scores_raw = bm25_retriever.vectorizer.get_scores(query.split())

    # Step 2: Vector similarity scores
    vector_results = vectorstore.similarity_search_with_score(query, k=10)

    # Convert to dict
    vector_scores_raw = {doc.page_content: score for doc, score in vector_results}

    # Step 3: Normalize scores
    def normalize(scores):
        min_s = min(scores)
        max_s = max(scores)
        return [(s - min_s) / (max_s - min_s + 1e-8) for s in scores]

    # Normalize BM25
    bm25_norm_scores = normalize(bm25_scores_raw)

    # Map BM25 scores to docs
    bm25_scores = {}
    for doc, score in zip(chunks, bm25_norm_scores):
        bm25_scores[doc.page_content] = score

    # Normalize vector scores (lower distance = better)
    if vector_scores_raw:
        vec_values = list(vector_scores_raw.values())
        vec_norm = normalize([-v for v in vec_values])  # invert distance
        vector_scores = dict(zip(vector_scores_raw.keys(), vec_norm))
    else:
        vector_scores = {}

    # Step 4: Combine scores
    combined = defaultdict(float)

    for text, score in bm25_scores.items():
        combined[text] += 0.4 * score

    for text, score in vector_scores.items():
        combined[text] += 0.6 * score

    # Step 5: Sort
    ranked = sorted(combined.items(), key=lambda x: x[1], reverse=True)

    # Step 6: Map back to documents
    doc_map = {doc.page_content: doc for doc in chunks}

    return [doc_map[text] for text, _ in ranked[:top_k]]

In [24]:
import cohere
os.environ["COHERE_API_KEY"] = os.getenv("COHERE_API_KEY")

# Cohere client
co = cohere.Client()


def rerank_documents(query, docs, top_n=3):
    doc_texts = [doc.page_content for doc in docs]

    response = co.rerank(
        model="rerank-english-v3.0",
        query=query,
        documents=doc_texts,
        top_n=top_n
    )

    return [docs[r.index] for r in response.results]

In [25]:
def hybrid_rag_with_rerank(query):
    # Step 1: Hybrid retrieval (more docs)
    hybrid_docs = hybrid_retrieve(query, top_k=10)

    # Step 2: Rerank (precision boost)
    final_docs = rerank_documents(query, hybrid_docs, top_n=3)

    # Step 3: Context
    context = "\n\n".join([doc.page_content for doc in final_docs])

    # Step 4: Strong prompt
    prompt = f"""
    You are a strict legal assistant.

    RULES:
    - Use ONLY the context
    - Do NOT add external knowledge
    - If answer not found, say "I don't know"

    Context:
    {context}

    Question:
    {query}

    Answer:
    """

    response = llm.invoke(prompt)

    return response.content, final_docs

In [27]:
question = "What is the rent amount and due date mentioned in the lease agreement?"
ans = hybrid_rag_with_rerank(question)
print(ans)

('The rent amount is Rs. 17,600/- (Rupees Six Thousand Six Hundred only) and it is due before the 07th day of each English calendar month.', [Document(metadata={}, page_content='AND WHEREAS THE BOTH THE PARTIES HAVE AGREED FOR IT ON THE FOLLOWING TERM AND CONDITIONS:-\n\n1. That the Lessee shall pay rent of Rs. 17,600/- (Rupees Six Thousand Six Hundred only) every month before 07th day of each English calendar month to the owner.\n\n2. That the Lessee’s security deposit of Rs.17,600/- will be refunded back to the Lessee after adjustment of electric and water charges if pending, after the furniture and fixtures are duly handed over in same condition to the Lessor.\n\n3. The Tenancy shall commence from 01/10/2025 to 31/08/2026 for 11 months.'), Document(metadata={}, page_content='Sure! Here’s the extracted text:\n\n---\n\n**RENT AGREEMENT**\n\nThis Rent Agreement is made at Gurugram on 05/01/2026 between:\n\n**Smti. Suresh Devi W/o Sh. Mahender Singh R/o RZ-687/A, Raj Nagar Part-1, Palam

In [14]:
import time

answers = []
contexts = []

for q in questions:
    ans, docs = hybrid_rag_with_rerank(q)

    answers.append(ans)
    contexts.append([doc.page_content for doc in docs])

    time.sleep(7)  # avoid Cohere 429 error

In [15]:
from datasets import Dataset

eval_data = {
    "question": questions,
    "answer": answers,
    "contexts": contexts,
    "ground_truth": ground_truths
}

dataset = Dataset.from_dict(eval_data)

In [24]:
questions[10]

'When is the security deposit returned?'

In [25]:
answers[10]

'The security deposit will be returned upon vacating, returning the keys to the Landlord, and termination of the contract according to other terms agreed. It will be held intact by the Landlord until at least thirty (30) working days after the Tenants have vacated the property, and then returned within 60 days after they have vacated, minus any necessary charges for damages or repairs, with a written explanation of deductions.'

In [26]:
contexts[10]

['late payment. If for any reason a check is returned or dishonored, all future rent payments will be cash or \nmoney order.\n7. SECURITY DEPOSIT:\nTenants hereby agree to pay a security deposit of $685 to be refunded upon vacating, returning the keys to \nthe Landlord and termination of this contract according to other terms herein agreed. This deposit will be \nheld to cover any possible damage to the property. No interest will be paid on this money and in no case',
 'will it be applied to back or future rent. It will be held intact by Landlord until at least thirty (30) working \ndays after Tenants have vacated the property. At that time Landlord will inspect the premises thoroughly \nand assess any damages and/or needed repairs. This deposit money minus any necessary charges for \nmissing/dead light bulbs, repairs, cleaning, etc., will then be returned to Tenant with a written explanation \nof deductions, within 60 days after they have vacated the property.',
 'the month, Tenant ag

In [27]:
ground_truths[10]

'It is returned within 60 days after vacating, minus any deductions.'

In [28]:
from openai import OpenAI
import os
from dotenv import load_dotenv
load_dotenv()

from ragas.embeddings import OpenAIEmbeddings
from ragas.llms import llm_factory
from ragas.metrics import Faithfulness, ContextPrecision, ContextRecall
from ragas import evaluate

client = OpenAI()

ragas_llm = llm_factory(
    "gpt-4o-mini",
    client=client
)

ragas_embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    client=client,
)

metrics = [
    Faithfulness(llm=ragas_llm),
    ContextPrecision(llm=ragas_llm),
    ContextRecall(llm=ragas_llm),
]

result = evaluate(
    dataset,
    metrics=metrics,
)

print(result)

C:\Users\Oliver\AppData\Local\Temp\ipykernel_15040\4043799150.py:8: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness, ContextPrecision, ContextRecall
C:\Users\Oliver\AppData\Local\Temp\ipykernel_15040\4043799150.py:8: DeprecationWarning: Importing ContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextPrecision
  from ragas.metrics import Faithfulness, ContextPrecision, ContextRecall
C:\Users\Oliver\AppData\Local\Temp\ipykernel_15040\4043799150.py:8: DeprecationWarning: Importing ContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Context

{'faithfulness': 0.9000, 'context_precision': 0.9167, 'context_recall': 0.9250}


#### 🧠 Why This Works (Deep Insight)

##### 1. Hybrid = better recall - Find more relevant docs
##### 2. Rerank = better precision - Pick best docs from them
